# Day 19 — Credit Card Fraud Detection

Train imbalanced classifiers on 284,807 European card transactions (Sep 2013), 492 frauds (0.17%), and compare recall, precision and the operational cost of false alarms across decision thresholds.

This notebook reads **processed CSVs only** — it never touches the 144 MB raw file, so it re-executes in CI without the cache.

In [ ]:
from pathlib import Path
import json
import pandas as pd

def _find_root() -> Path:
    """Project root from the notebook file's own location."""
    here = Path.cwd()
    for candidate in [here, *here.parents]:
        if (candidate / "data" / "processed" / "model_metrics.csv").exists():
            return candidate
    raise FileNotFoundError("data/processed/model_metrics.csv not found")

ROOT = _find_root()
PROCESSED = ROOT / "data" / "processed"
source = json.loads((PROCESSED / "data_source.json").read_text())
print(f"Rows: {source['rows']:,}  frauds: {source['fraud']} "
      f"({source['fraud_rate_pct']}%)  synthetic: {source['synthetic']}")
print(f"Test set: {source['test_rows']:,} rows, {source['test_fraud']} frauds")

## 1. The imbalance

Fraud is 0.17% of transactions. Any model judged on accuracy alone can score 99.83% by never flagging anything — and catch nothing.

In [ ]:
class_summary = pd.read_csv(PROCESSED / "class_summary.csv")
amount_summary = pd.read_csv(PROCESSED / "amount_summary.csv")
display(class_summary)
display(amount_summary)

## 2. Model comparison (threshold 0.50, test set)

PR-AUC is the headline metric on a 0.17% problem; ROC-AUC is reported for completeness. Note the reversal: logistic regression has the higher ROC-AUC, random forest the higher PR-AUC — the two curves rank the models differently, which is exactly why PR-AUC is the right headline here.

In [ ]:
metrics = pd.read_csv(PROCESSED / "model_metrics.csv")
display(metrics[["model", "precision", "recall", "f1", "accuracy", "pr_auc", "roc_auc", "total_cost_usd"]])

## 3. Threshold tradeoff

Lowering the threshold catches more fraud (recall up) but flags more legitimate transactions (precision down, review cost up). The sweep below is for the best model by PR-AUC.

In [ ]:
sweep = pd.read_csv(PROCESSED / "threshold_sweep.csv")
best = source["best_model"]
best_sweep = sweep[sweep["model"] == best]
display(best_sweep[["threshold", "precision", "recall", "f1", "fp", "fn", "cost_per_10k_txns_usd"]])
opt = best_sweep.loc[best_sweep["cost_per_10k_txns_usd"].idxmin()]
print(f"Cost-optimal threshold for {best}: {opt['threshold']:.2f} — "
      f"${opt['cost_per_10k_txns_usd']:,.2f} per 10k txns, "
      f"precision {opt['precision']:.3f}, recall {opt['recall']:.3f}")

## 4. Precision–recall curves

In [ ]:
import matplotlib.pyplot as plt

curves = pd.read_csv(PROCESSED / "pr_curves.csv")
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot([0, 1], [0.0017, 0.0017], "--", color="grey", label="No-skill (prevalence 0.17%)")
for model, group in curves.groupby("model"):
    pr_auc = metrics.loc[metrics["model"] == model, "pr_auc"].iloc[0]
    ax.plot(group["recall"], group["precision"], label=f"{model} (PR-AUC {pr_auc:.3f})")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title("Precision–recall curves, test set")
ax.legend(); ax.grid(alpha=0.3)
plt.show()

## 5. Cost vs. threshold

Missed fraud costs its Amount; a false alarm costs $5.00 of manual review (assumption, labelled in data_source.json). The minimum is the cost-optimal operating point.

In [ ]:
cost = pd.read_csv(PROCESSED / "cost_curve.csv")
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(cost["threshold"], cost["cost_per_10k_txns_usd"], marker="o")
ax.axvline(source["best_threshold"], color="grey", linestyle="--")
ax.set_xlabel("Decision threshold")
ax.set_ylabel("Expected cost per 10,000 txns (USD)")
ax.set_title(f"Cost vs. threshold — {source['best_model']}")
ax.grid(alpha=0.3)
plt.show()

## 6. Verification

One number checked by hand: the class summary must reproduce the published fingerprint — 284,807 rows, 492 frauds.

In [ ]:
assert int(class_summary["n_transactions"].sum()) == 284_807
assert int(class_summary.loc[class_summary["Class"] == 1, "n_transactions"].iloc[0]) == 492
print("Fingerprint verified: 284,807 rows, 492 frauds (0.1727%)")

## Findings

1. **Random forest wins on PR-AUC** (the headline metric for a 0.17% problem); logistic regression wins on ROC-AUC — the two rankings disagree, so the metric choice matters.
2. **At threshold 0.50 the forest is precise but leaky**: precision 0.97, recall 0.70 — it almost never cries wolf, but misses ~30% of frauds. Logistic regression catches 88% of frauds but flags 1,806 legitimate transactions to do it.
3. **Dropping the forest's threshold to 0.10 is cheaper overall**: recall rises to ~0.84 while the cost per 10,000 transactions falls to the sweep minimum — missed-fraud dollars saved outweigh the extra $5 reviews.

**Recommendation:** deploy the random forest at threshold 0.10 for the alert queue, and re-tune the threshold on a separate validation set (not the test set) before production use.